# Dictionary Task Control Processing [DTCP]

## Name Description

### The goal is to specify a complete work description ***wrk_task*** by selecting ***wrk_tsk: key***

#### Sturcture
1. That will select data for parameter(i) ***dat_dpl(i)*** from any ***dat_nm*** in any ***dat_root_dct*** of the form ***[dat_root_dct: dat_key_nm]***
2. Repeat for all parameters (all i)
3. To be processed ***prcssr_dpl*** by any ***prcssr_mode*** of any processor ***prcssr_nm*** of the form ***[prcssr_nm_dct: prcssr_key_nm]***
4. The output destination ***rslts_dpl*** in ***rslts_typ*** of rslts ***rslts_nm*** of the form ***[rslts_nm_dct: rslts_key_nm]***
5. ***wrk_tsk*** = all(i):***dat_dpl(i)*** + ***prcssr_dpl*** + ***rslts_dpl***
6. Each ***wrk_tsk***  is tested, classified, dated, tagged, then added  to the ***wrk_tsk_dct***
wrk_tsk5. ***
7. Thrn
8. 

# things to add

## list
1.  Add visiblity "vbl" as opttion to "CLS" "dtv" chice  col
2.  there is a "visibility reset" that makes ***all col visible***
3.  Macro to hide any col where the cell in the rowID is ***double clicked row*** is blank
4.  A note can be inserted in any row to make a row visible when that row id is "doulble clicked"
5.  Macro that will show all metro cols plus all cols that have the same value in the same row As a cell that is***shift douple clicked***
6. Any note will cause the col to be shown so info can be inserted
7.An attribute col is placed in the col adj to the cls col
8. A macro is designsed so when the cls in a row is double clicked a drop down box will show the drop box  and the selected one will be added to the attribute col.
9. Therefore shift double clicking class the value in the class col will make all cols with that class be visible
10. Then double clicking the attriute in the attribute col will further filter the visible to caos with that class and that attribute

## Design a unified structure that
1. can support different data types
2. required by a variety of rootx
3. to be used in conjunction with other ***"rootx_dct"***
4. to create diverse working data ***duple***  *named* "***wrk_dpl_z***"

## Designed to be used in conjunction with other ***"rootx_dct"*** to create diverse working data ***duple***  *named* ***wrk_dpl_z***

1. a combination of
2. ....***"root_x_dcts"***
3. ....that has ***"keys"*** = "dat_col_nm_y"
4. ....and "***"rows"*** with similar criterea
5. to *uniquely and cleanly specify* the ***duple***
6. ....*specifying* any ***dat_col_y***
7. ....from any ***rootx*** 
8. ....that need to be processed together
9. *named* ***wrk_dpl_z***

## Creating the *wrk_dpl_z*  for ***mulit dat_col analysis***.

### Collecting the root Data, adding it to root_table and moving root_tabl to WSL folder

1. Each ***rootx*** has the *same unique data cols set* and *row critera* [ie ib770 data columns and each row is a test]
2. ....The data from each test of ***rootx*** are stored as a  *timestamped CSV file* in a folder named "***rootx***
3. The rootx data is accumulated in an XLTable called ***rootx_tbl***
4. The **rootx_tbl*** is a *multicol XLworkbook table* in a XLworkbook located in ***"onedrive personal"*** named ***dat_mnl***
5. It is populated by importing the cvs files folder ***rootx*** that do not have a prefix "z_"
6. ....Once imported each "csv file is prefixed with "z_" showing that it has been imported.
7. ....Once imported, each "csv file is prefixed with "z_," showing that it has been imported.
8. ....It is unchanged so it remains as a truth file for that record.
9. Then the XLworkbook located in ***"onedrive personal"*** named ***dat_mnl***
10. ....is copied and pasted into the WSL folder "\\wsl.localhost\Ubuntu-20.04\home\ratlabs\JL_2\data\dat_mnl\dat_mnl.xlsm"
11. ....This is performed by a notepad batch file "copy_dat_mnl"

### Creating the ***root_dct*** by importing the updated ***rootx_tbl***

### Creating the ***wrk_dpl_z*** by importing the ***wrk_dpl_tbl*** and specifying a ***tsk***

1. 
11. using macro "***rootx1_[*key:dat_col*]***,
12. The list of *dat_cols* to be *worked on* is specified by a list ***rootx1_[*key:dat_col_y1*]***,***rootx2_[*key:dat_col_y2*]***,......

# New def functions for creating root_dct

### Importing a root table from XL and creating root_dct in Jupyter

In [7]:
def import_root(root: str):
    from pathlib import Path
    import pandas as pd

    """
    General-purpose importer for Excel files with:
      - filename: <root>.xlsm
      - sheet:    <root>_main
      - header row detected by the presence of 'col_nms'
    """

    # Build path: /home/ratlabs/JL_2/data/<root>/<root>.xlsm
    xl_path = Path(f"/home/ratlabs/JL_2/data/{root}/{root}.xlsm")

    # Raw import (no header)
    df_raw = pd.read_excel(
        xl_path,
        sheet_name=f"{root}_main",
        header=None,
        engine="openpyxl"
    )

    # Detect header row containing "col_nms"
    header_row = df_raw.index[df_raw.eq("col_nms").any(axis=1)][0]

    # Promote that row to header
    df = df_raw.copy()
    df.columns = df.iloc[header_row].astype(str)

    # Drop rows up to and including header row
    df = df.drop(index=range(header_row + 1)).reset_index(drop=True)

    return df_raw, df


#### Why this is useful
1. Single function handles all your XL imports
2. Root‑based naming convention keeps your folder structure clean
3. Header detection logic stays identical
4. Zero hard‑coding → fewer brittle paths
5. Operator‑grade determinism: predictable, reproducible, minimal moving parts

### Using pickle re: retrieve and store post_edit

#### You want two functions:
1. df_to_dct(df_nm) → read df_nm.pkl, convert to dict, write nm_dct.pkl
2. dct_frm_pkl(nm_dct) → read nm_dct.pkl and return the dictionary

In [8]:
def df_to_dct(df_nm: str) -> dict:
    import pickle
    import pandas as pd
    from pathlib import Path

    """
    Load a DataFrame from df_nm.pkl, convert it to a dictionary,
    write nm_dct.pkl, and return the dictionary.
    """
    repo_dir = Path(__file__).resolve().parent  # repo directory
    df_path = repo_dir / f"{df_nm}.pkl"
    dct_path = repo_dir / "nm_dct.pkl"

    # Load DataFrame
    df = pd.read_pickle(df_path)

    # Convert to dictionary (records format preserves rows cleanly)
    nm_dct = df.to_dict(orient="records")

    # Write dictionary to pickle
    with open(dct_path, "wb") as f:
        pickle.dump(nm_dct, f)

    return nm_dct


def dct_frm_pkl(nm_dct: str = "nm_dct") -> dict:
    """
    Load nm_dct.pkl and return the dictionary.
    """
    repo_dir = Path(__file__).resolve().parent
    dct_path = repo_dir / f"{nm_dct}.pkl"

    with open(dct_path, "rb") as f:
        recovered_dct = pickle.load(f)

    return recovered_dct


🧭 Notes for Operator Reliability
df_nm is the base name (no extension).
Example: df_to_dct("df_nm") loads df_nm.pkl.

Dictionary format uses records → each row becomes a dict.

Repo directory is inferred from the script’s location (__file__).

Both pickle files live in the same directory as the script.

### Discription written in raw of the Macro used to read in the individual rows in a folder to the "root table in XL

### next

## Def functions

In [2]:
# def import_dat_mnl():                                  #Fixed Path  Called Below Returns df_raw, df
import pandas as pd
from pathlib import Path

def import_dat_mnl():
    # Raw import (no header)
    xl_path = Path("/home/ratlabs/JL_2/data/dat_mnl/dat_mnl.xlsm")
    df_raw = pd.read_excel(
        xl_path,
        sheet_name="dat_mnl_main",
        header=None,
        engine="openpyxl"
    )

    # Detect the real header row by searching for "col_nms"
    header_row = df_raw.index[df_raw.eq("col_nms").any(axis=1)][0]

    # Promote that row to header
    df = df_raw.copy()
    df.columns = df.iloc[header_row].astype(str)

    # Drop all rows up to and including the header row
    df = df.drop(index=range(header_row + 1)).reset_index(drop=True)

    return df_raw, df



In [3]:
# OLD VERSION def write_df_to_pickle(df, filename):
def write_df_to_pickle(df, filename):
    """
    Writes a DataFrame to a pickle file.

    Parameters
    ----------
    df : pd.DataFrame
        The dataframe to save.
    filename : str
        The pickle filename, e.g. 'mydata.pkl'.
    """
    df.to_pickle(filename)

# usage 
# write_df_to_pickle(df, "df.pkl")

In [4]:
# OLD VERSION def load_df_from_pickle(filename):

def load_df_from_pickle(filename):
    """
    Loads a DataFrame from a pickle file.

    Parameters
    ----------
    filename : str
        Path to the pickle file.

    Returns
    -------
    pd.DataFrame
    """
    return pd.read_pickle(filename)

    # usage 
    # df = load_df_from_pickle("df.pkl")



In [48]:
def fix_manual_df(df):
    """
    Converts a raw df where:
      - row 0 = column numbers
      - row 1 = actual column names
      - row 2+ = data
    into a clean dataframe with correct headers.
    """

    # Extract row 1 as header
    new_cols = df.iloc[1].tolist()

    # Apply new header
    df_fixed = df.copy()
    df_fixed.columns = new_cols

    # Drop row 0 and row 1
    df_fixed = df_fixed.drop(index=[0, 1]).reset_index(drop=True)

    return df_fixed


In [78]:
def create_plt_lst(dat_col_dct, ib_dct, plt_active_lst):
    """
    Filters dat_col_dct so that:
      - Only columns listed in plt_active_lst are included
      - Each column is filtered to rows whose dtv matches df_77_97_mrn['dtv']

    Returns:
        plt_active_dct : dct
            Keys = column names in plt_active_lst
            Values = filtered pandas Series aligned by dtv
    """

    # Extract the dtv values we want to keep
    dtv_filter_values = set(ib_dct["dtv"].unique())

    plt_active_dct = {}

    for col in plt_active_lst:
        if col not in dat_col_dct:
            continue  # skip missing columns safely

        series = dat_col_dct[col]

        # Filter the series by dtv alignment
        # Assumes dat_col_dct["dtv"] exists and is aligned row‑wise
        dtv_series = dat_col_dct["dtv"]

        filtered_series = series[dtv_series.isin(dtv_filter_values)]

        plt_active_dct[col] = filtered_series.reset_index(drop=True)

    return plt_active_dct


## importing the dat_mnl

## Building dat_mnl_dct

### Update ***"df_dat_mnl"*** from ***"XL"***

In [107]:
df_dat_mnlx,df_dat_mnl= import_dat_mnl()               # Load from XL 

In [108]:
# verify df_dat_mnl   #works

In [109]:
# verify df_dat_mnlx   #works

In [110]:
# verify type(df_dat_mnl)

In [111]:
# verify 
df_dat_mnl.columns.tolist     # Works

<bound method IndexOpsMixin.tolist of Index(['col_nms', 'tst#', 'dtv', 'timestamp', 'Notes', 'urinePH1_5',
       'bullet coffee', 'keto_1', 'Stamina1_5', 'bd_leg_heat',
       ...
       'Column186', 'Column187', 'Column188', 'Column189', 'Column190',
       'Column191', 'Column192', 'Column193', 'Column194', 'urine smell'],
      dtype='object', name=1, length=226)>

### Edit the ***"cstm_dat_mnl"*** list

In [84]:
df_dat_mnlx = fix_manual_df(df_dat_mnlx)        # Look to 2nd row for column names the numbers are in the first row



In [131]:
# verify df_dat_mnlx

# Creating a plt_lst of a dictionary of dat_mnl cols as the keys that have data in rows "dtv"
1. matches "df_77_97_mrn" "dtv rows"
2. and contains desired "dat_mnl dat_cols"
3. It will be stored in "dat_mnl dct" pkl and used in plot def functions along with  "ib_dct" "dtv rows"

In [113]:
dat_col_dct = {col: df_dat_mnl[col] for col in df_dat_mnl.columns} # Calc the keys to the use for 


In [12]:
# verify list(dat_col_dct.keys())             # worked


In [10]:
# verify dat_col_dct    #worked

In [88]:
dtv = dat_col_dct["dtv"]
# verify   dtv  # worked has all days

In [89]:
notes = dat_col_dct["Notes"]

In [90]:
# verify notes           #works

## Write the ***"dat_col_dct"*** to Pickle so it can be used to create plot by going down dictionaries

In [118]:
import pickle
with open("df_77_97_mrn.pkl", "rb") as f:  
    df_77_97_mrn = pickle.load(f)

In [119]:
# verify df_77_97_mrn #Works

In [120]:
plt_active_lst = ["dtv", "timestamp", "Notes"]  # 


In [121]:
plt_active_lst


['dtv', 'timestamp', 'Notes']

In [122]:
print(df_77_97_mrn.columns.tolist())


['timestamp', 'dtv', 'weight', 'vfa_(visceral_fat_area)', 'ecw/tbw', 'ecw/tbw_of_left_leg_x', 'ecw/tbw_of_right_leg_x', 'bmr_(basal_metabolic_rate)', 'smm_(skeletal_muscle_mass)', 'khz-whole_body_phase_angle', 'whole_body_ecw/tbw_t_score', 'ecw_(extracellular_water)', 'icw_(intracellular_water)', 'ecw/tbw_of_left_leg_y', 'ecw/tbw_of_right_leg_y', 'ecw_of_left_leg', 'ecw_of_right_leg', 'lower_limit_(ecw_of_left_leg_normal_range)', 'lower_limit_(ecw_of_right_leg_normal_range)', 'upper_limit_(ecw_of_left_leg_normal_range)', 'upper_limit_(ecw_of_right_leg_normal_range)']


In [13]:
# plt_active_dct = (dat_col_dct, df_77_97_mrn, plt_active_lst)

In [129]:
# verify plt_active_dct["dtv"]                    # worked
# verify plt_active_dct["timestamp"]             # worked
# verify plt_active_dct["slp_hr"]  

In [126]:
for i, col in enumerate(df_77_97_mrn.columns):
    print(i, repr(col))


0 'timestamp'
1 'dtv'
2 'weight'
3 'vfa_(visceral_fat_area)'
4 'ecw/tbw'
5 'ecw/tbw_of_left_leg_x'
6 'ecw/tbw_of_right_leg_x'
7 'bmr_(basal_metabolic_rate)'
8 'smm_(skeletal_muscle_mass)'
9 'khz-whole_body_phase_angle'
10 'whole_body_ecw/tbw_t_score'
11 'ecw_(extracellular_water)'
12 'icw_(intracellular_water)'
13 'ecw/tbw_of_left_leg_y'
14 'ecw/tbw_of_right_leg_y'
15 'ecw_of_left_leg'
16 'ecw_of_right_leg'
17 'lower_limit_(ecw_of_left_leg_normal_range)'
18 'lower_limit_(ecw_of_right_leg_normal_range)'
19 'upper_limit_(ecw_of_left_leg_normal_range)'
20 'upper_limit_(ecw_of_right_leg_normal_range)'


In [106]:
# This is the ready to plot list of combined 97 77 data of most interest
import pickle

write_df_to_pickle(df_77_97_mrn, "df_77_97_mrn.pkl")
print("df_77_97_mrn written to pickle")
# verify df_77_97_mrn

df_77_97_mrn written to pickle
